# 03 — Weather Feature Analysis & Fire Risk Correlations

Analyzes the relationship between environmental conditions (ERA5 weather reanalysis) and fire occurrence.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/processed/train.parquet")
df = pd.read_parquet(DATA_PATH)
print(f"Loaded training split with {len(df):,} rows and {len(df.columns)} columns.")


## 1. Temperature Distributions: Fire vs. No-Fire Days


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.kdeplot(data=df, x="temp_max", hue="fire_occurred", common_norm=False, fill=True, alpha=0.4, ax=ax)
ax.set_title("Max Temperature Distribution on Fire (1) vs No-Fire (0) Days", fontsize=13, fontweight="bold")
ax.set_xlabel("Max Daily Temperature (°C)")
plt.tight_layout()
plt.show()


## 2. Rolling 30-Day Precipitation vs. Fire Occurrence


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="fire_occurred", y="precip_sum_30d", palette="Set2", showfliers=False, ax=ax)
ax.set_title("30-Day Rolling Precipitation (mm) vs Fire Occurrence", fontsize=13, fontweight="bold")
ax.set_xticklabels(["No Fire (0)", "Fire (1)"])
ax.set_ylabel("Rolling 30-Day Precip (mm)")
plt.tight_layout()
plt.show()


## 3. Weather Feature Correlations with Target


In [ ]:
weather_cols = [c for c in df.columns if any(w in c for w in ['temp', 'precip', 'wind', 'et0', 'dry'])]
corr = df[weather_cols + ['fire_occurred']].corr()['fire_occurred'].sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
corr.drop('fire_occurred').plot(kind='barh', ax=ax, color=np.where(corr.drop('fire_occurred')>0, '#e74c3c', '#3498db'))
ax.set_title("Correlation of Weather Features with Fire Occurrence Target", fontsize=13, fontweight="bold")
ax.set_xlabel("Pearson Correlation Coefficient")
plt.tight_layout()
plt.show()


## 4. Key Findings Summary


- **Temperature & Dryness**: `temp_max`, `et0` (evapotranspiration), and length of `dry_days_streak` correlate positively with wildfire ignition.
- **Precipitation Safeguard**: `precip_sum_7d`, `precip_sum_14d`, and `precip_sum_30d` show strong negative correlations, confirming that recent rainfall significantly reduces fire probability.
